# Lab 7: Estimation & Bootstrapping — How Uncertain Is the Median?
## ECON 3916: Applied Machine Learning for Economists
### Guided Construction Lab | 40 min Core + 15 min AI + 15 min Extension + take-home Challenge

---

**Learning Objectives:**
- Distinguish the standard deviation of the data from the standard error of an estimate
- Build a parametric confidence interval from the CLT and state what the 95% refers to
- Run the bootstrap by hand, then with `scipy.stats.bootstrap`
- Recognise when a bootstrap interval has collapsed, and explain why that is the method working rather than failing
- Check a bootstrap function written by AI against the intervals you computed yourself

**Dataset:** UCI Adult Census Income — 32,561 respondents, 1994 Current Population Survey extract

**Prerequisites:** the Part 0 sections of Labs 1-5, and the labs before this one.

**How this lab works:** Parts 1-3 are GUIDED: run each cell as it is, then answer the questions. Part 4 runs one guided step, then has two short blanks to fill in. The Extension is optional, and the Challenge is a take-home. The AI-Assisted Expansion at the end is the one place you use AI to write code, and you check what it writes against your own numbers.

**AI in the manual parts:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion while you work on those parts. AI-assisted coding is allowed only where this lab says so.

---

> ### What you write, and what you run
>
> **You write:**
> - **Part 4, Step 11:** two blanks (`____`), one line each: what `mean_difference()` returns, and the statistic you hand to `bootstrap`.
> - The answers to the interpretation questions after each part, in text cells.
> - **The AI Expansion:** version 2 of the given prompt, revised once, after you find what version 1 left the AI to guess. Then four short answers: your revision, the check against your own Step 8 and Step 10 intervals, the key lines in your own words, and when AI is worth using for this.
>
> **You run and read:** everything else. Setup, Parts 1-3, Step 10, the Extension, and the check cell in the AI Expansion are working code to learn from. Read each cell before you run it, and make sure you can say what each line does. The AI's function is pasted, not typed, and the code itself is not graded; your answers about it are.
>
> **The Challenge** (take-home) has two one-line blanks in `ratio_90_10()` and a short written reflection. The **README** is text only: the prompt at the end is for documentation, not code.

## Setup

In [ ]:
# GUIDED — run as-is
# Step 1: import libraries and fix the seed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from scipy.stats import bootstrap

RANDOM_STATE = 42
# One generator for Parts 1-3. Each cell carries on its sequence, so run the cells in order to get these numbers
rng = np.random.default_rng(seed=RANDOM_STATE)

# savefig will not create a folder, and a fresh Colab session has no figures/ folder
Path("figures").mkdir(exist_ok=True)

print("Setup complete.")

## Part 1: Standard Deviation vs. Standard Error (GUIDED — 10 min)

Chapter 7 opened with a distinction that trips up experienced analysts. The **standard
deviation** measures how spread out the *observations* are. The **standard error** measures
how spread out the *estimate* would be across repeated samples. They answer different
questions, and only one of them shrinks as you collect more data.

In [ ]:
# GUIDED — run as-is
# Step 2: load the UCI Adult Census Income data
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
# The file has no header row, so the column names are listed here
columns = ["age", "workclass", "fnlwgt", "education", "education_num",
           "marital_status", "occupation", "relationship", "race", "sex",
           "capital_gain", "capital_loss", "hours_per_week",
           "native_country", "income"]
# skipinitialspace drops the space after each comma, so " Male" is read as "Male"
df = pd.read_csv(url, names=columns, skipinitialspace=True)

print(f"Shape: {df.shape}")
print(f"Median age:          {df['age'].median():.0f}")
print(f"Median hours/week:   {df['hours_per_week'].median():.0f}")
# A True/False column's mean is the share that is True
print(f"Share with any capital gain: {(df['capital_gain'] > 0).mean():.1%}")
df[["age", "hours_per_week", "capital_gain"]].head()

In [ ]:
# GUIDED — run as-is
# Step 3: treat the full file as a population, then sample from it
population = df["age"].values      # .values gives the column as a NumPy array
n = 100
n_samples = 1_000

# [... for _ in range(n_samples)] repeats the draw 1,000 times and collects each sample's mean.
# replace=False: nobody is drawn twice within one sample, as in a survey
sample_means = np.array([rng.choice(population, size=n, replace=False).mean()
                         for _ in range(n_samples)])

# ddof=1 divides by n - 1, the usual sample standard deviation
print(f"Population SD of age:            {population.std(ddof=1):>7.3f} years")
print(f"SE of the mean (empirical):      {sample_means.std(ddof=1):>7.3f} years")
print(f"SE of the mean (formula s/sqrt n):{population.std(ddof=1) / np.sqrt(n):>7.3f} years")

In [ ]:
# GUIDED — run as-is
# Step 4: the sqrt(n) payoff. Quadruple n, halve the SE
print(f"{'n':>8} {'SE (empirical)':>16} {'SE (formula)':>14}")
print("-" * 40)      # a line of 40 dashes
for size in [25, 100, 400, 1_600]:
    means = np.array([rng.choice(population, size=size, replace=False).mean()
                      for _ in range(500)])
    formula = population.std(ddof=1) / np.sqrt(size)
    print(f"{size:>8,} {means.std(ddof=1):>16.3f} {formula:>14.3f}")

**Expected output:** the SD of age is 13.640 years. It describes people, and it is a fact about
the population, whatever sample you draw. The SE describes the estimate of the average: 1.361
years across 1,000 samples of 100, against 1.364 from the formula. In the table, each 4x in
sample size halves the SE, from 2.583 at n = 25 to 0.337 at n = 1,600.

### Interpretation Questions

1. The SD of age is one fixed number (13.640), but the SE column falls as size grows. Explain in one sentence why only the SE depends on sample size.
2. A colleague reports "average age 38.6, standard deviation 13.6" and concludes the estimate is imprecise. What have they confused, and what number should they have quoted instead?
3. You need to halve your current standard error. By what factor must the sample grow?

*Your answers here:*

## Part 2: The Parametric Confidence Interval (GUIDED — 10 min)

The CLT says the sampling distribution of the mean approaches normality regardless of the
shape of the underlying data, *given enough observations*, and how many is enough depends
on the skew.

In [ ]:
# GUIDED — run as-is
# Step 5: a textbook parametric CI for mean age
sample = rng.choice(population, size=200, replace=False)

xbar = sample.mean()
se = sample.std(ddof=1) / np.sqrt(len(sample))
# 1.96 standard errors either side of the mean; ci holds two numbers, ci[0] the lower end and ci[1] the upper
ci = (xbar - 1.96 * se, xbar + 1.96 * se)

print(f"Sample mean:      {xbar:.2f} years")
print(f"Standard error:   {se:.3f}")
print(f"95% CI:           [{ci[0]:.2f}, {ci[1]:.2f}]")
print(f"True population mean: {population.mean():.2f}")

**Expected output:** a sample mean of 38.34 years, a standard error of 0.941, and a 95% CI of
[36.49, 40.18]. The true population mean, 38.58, is inside this interval. Question 1 below asks
what the 95% promises about intervals in general.

In [ ]:
# GUIDED — run as-is
# Step 6: how fast does the CLT kick in? Age is mildly skewed; capital gains are far more skewed
# .loc[rows, column]: the capital_gain column, for the rows where it is above 0
cap_gain = df.loc[df["capital_gain"] > 0, "capital_gain"].values

print(f"Skewness of age:                  {stats.skew(population):>6.2f}")
print(f"Skewness of positive capital gain:{stats.skew(cap_gain):>6.2f}")

# A 2 x 3 grid of panels: one row per variable, one column per sample size
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
# enumerate counts as it loops (row is 0, then 1); each pair unpacks into data and label
for row, (data, label) in enumerate([(population, "age"),
                                     (cap_gain, "capital gain > 0")]):
    for col, size in enumerate([5, 30, 500]):
        means = np.array([rng.choice(data, size=size, replace=True).mean()
                          for _ in range(2_000)])
        axes[row, col].hist(means, bins=40, color="steelblue", edgecolor="white")
        axes[row, col].set_title(f"{label}, n = {size}")

plt.suptitle("Sampling distribution of the mean as n grows")
plt.tight_layout()
plt.savefig("figures/ch07_clt_convergence.png", dpi=150, bbox_inches="tight")
plt.show()

**Expected output:** a skewness of 0.56 for age and 3.41 for positive capital gains. In the
figure, the top row (age) is bell-shaped by n = 30. The bottom row is still visibly
right-skewed at n = 30: for capital gains, the CLT has not arrived yet.

### Interpretation Questions

1. "A 95% confidence interval has a 95% probability of containing the true mean." Chapter 7 says this is wrong. State the correct interpretation, and say what the 95% is a property of.
2. From the figure: at which sample size would you trust a parametric CI for mean age? For mean capital gain? Justify from the plots, not from the n > 30 rule.
3. The parametric CI formula needs `s / sqrt(n)`. Name one statistic in this dataset for which no such formula exists.

*Your answers here:*

## Part 3: The Bootstrap by Hand (GUIDED — 10 min)

The bootstrap replaces the missing formula with simulation: resample your own data *with
replacement*, recompute the statistic, and let the spread of those recomputed values stand in
for the sampling distribution.

Watch the first one collapse. That is not a bug, and understanding why is the point of this part.

In [ ]:
# GUIDED — run as-is
# Step 7: bootstrap the median age, and watch it degenerate
ages = df["age"].values
B = 2_000      # the number of bootstrap resamples

# Each resample draws len(ages) people WITH replacement from the data itself, then takes the median
boot_medians = np.array([np.median(rng.choice(ages, size=len(ages), replace=True))
                         for _ in range(B)])

print(f"Sample median age: {np.median(ages):.1f}")
print(f"Bootstrap SE:      {boot_medians.std(ddof=1):.4f}")
# The middle 95% of the bootstrap medians: from the 2.5th to the 97.5th percentile
print(f"Bootstrap 95% CI:  [{np.percentile(boot_medians, 2.5):.1f}, "
      f"{np.percentile(boot_medians, 97.5):.1f}]")
# np.unique keeps one copy of each different value
print(f"Distinct bootstrap medians: {len(np.unique(boot_medians))}")

**Expected output:** a sample median of 37.0, a bootstrap SE of 0.0000, an interval of
[37.0, 37.0], and 1 distinct bootstrap median. Every resample returns the same median. Ages are
whole numbers, and 858 of the 32,561 people are exactly 37, so no resample moves the middle of
the line off that age. The bootstrap is correctly reporting that there is no variation to find:
it cannot manufacture what is not there.

In [ ]:
# GUIDED — run as-is
# Step 8: now a statistic where the bootstrap earns its keep.
# The same loop as Step 7, with the mean of capital gains in place of the median age
boot_means = np.array([rng.choice(cap_gain, size=len(cap_gain), replace=True).mean()
                       for _ in range(B)])

# >10,.0f right-aligns the number in 10 characters, with commas and no decimals
print(f"n (positive capital gain): {len(cap_gain):,}")
print(f"Mean:   ${cap_gain.mean():>10,.0f}")
print(f"Median: ${np.median(cap_gain):>10,.0f}")
print()
print(f"Bootstrap SE of the mean:  ${boot_means.std(ddof=1):>8,.0f}")
print(f"Bootstrap 95% CI (percentile): "
      f"[${np.percentile(boot_means, 2.5):,.0f}, ${np.percentile(boot_means, 97.5):,.0f}]")

**Expected output:** 2,712 people report a positive capital gain. Their mean is \$12,939 and
their median \$7,298: a long right tail pulls the mean far above the median. The bootstrap SE
of the mean is \$424, and the percentile interval is [\$12,114, \$13,780]. The AI Expansion at
the end of the lab comes back to this interval.

In [ ]:
# GUIDED — run as-is
# Step 9: the two distributions students most often confuse
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(cap_gain, bins=50, color="peru", edgecolor="white")
axes[0].axvline(cap_gain.mean(), color="black", linestyle="--",
                label=f"mean = ${cap_gain.mean():,.0f}")
axes[0].set_title("The DATA: capital gains of those who report any")
axes[0].set_xlabel("Capital gain ($)")
axes[0].legend()

axes[1].hist(boot_means, bins=50, color="steelblue", edgecolor="white")
axes[1].set_title(f"The ESTIMATE: {B:,} bootstrap means")
axes[1].set_xlabel("Bootstrap mean ($)")

plt.tight_layout()
plt.savefig("figures/ch07_bootstrap_vs_data.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Data spread (SD):          ${cap_gain.std(ddof=1):>9,.0f}")
print(f"Estimate spread (boot SE): ${boot_means.std(ddof=1):>9,.0f}")

**Expected output:** the data's SD is \$22,395 and the bootstrap SE is \$424, about 53 times
smaller. That is close to √2,712 ≈ 52, the square root of n. The right panel shows the
precision of the estimate, not the spread of the data.

### Interpretation Questions

1. Step 7 produced a bootstrap SE of 0.0000. Is the median age therefore known with certainty? Explain what the number actually says.
2. Why does resampling **with** replacement create variability, when resampling *without* replacement would not?
3. A reader glances at the right-hand panel of Step 9 and concludes "most people's capital gains are around \$13,000." What have they misread?

*Your answers here:*

## Part 4: scipy.stats.bootstrap and Two-Sample Comparisons (YOUR TASK — 10 min)

The manual loop is clear but slow. In practice you would use `scipy.stats.bootstrap`, which also
gives you **BCa** intervals — bias-corrected and accelerated, the method Chapter 7 tells you
to reach for by default. Step 10 is guided; Step 11 has two blanks.

In [ ]:
# GUIDED — run as-is
# Step 10: percentile vs BCa on the same statistic
# (cap_gain,) is a tuple holding one array: scipy takes a tuple because some statistics need two samples (Step 11).
# batch=100 draws 100 resamples at a time instead of all 2,000 at once, which saves memory.
# Each method gets a fresh generator with the lab's seed, so both see the same resamples
for method in ["percentile", "BCa"]:
    res = bootstrap((cap_gain,), np.mean, n_resamples=B, batch=100,
                    method=method, confidence_level=0.95,
                    random_state=np.random.default_rng(RANDOM_STATE))
    lo, hi = res.confidence_interval      # the two ends of the interval
    print(f"{method:>10}: [${lo:>9,.0f}, ${hi:>9,.0f}]   width = ${hi - lo:>8,.0f}")

**Expected output:** percentile [\$12,173, \$13,796], width \$1,623; BCa [\$12,180, \$13,809],
width \$1,629. BCa moves both ends up, by \$7 and \$13, to correct for skew in the bootstrap
distribution. On a symmetric statistic the two methods agree; the more skewed it is, the more
they differ.

The percentile interval here is not quite Step 8's [\$12,114, \$13,780], although both are
2,000-resample percentile intervals for the same mean. They drew different random resamples,
and that alone moves the ends by tens of dollars. The AI Expansion uses both.

In [ ]:
# YOUR TASK — fill in the two blanks (____)
# Step 11: Do men and women in this sample report different weekly hours?
men = df.loc[df["sex"] == "Male", "hours_per_week"].values
women = df.loc[df["sex"] == "Female", "hours_per_week"].values

print(f"Men:   n = {len(men):>6,}  mean = {men.mean():.2f} hours")
print(f"Women: n = {len(women):>6,}  mean = {women.mean():.2f} hours")
print(f"Observed difference: {men.mean() - women.mean():.2f} hours")

# Blank 1: the difference in means, men minus women. scipy hands each resampled group
# to this function as a separate argument: a is a resample of men, b a resample of women
def mean_difference(a, b):
    return ____

# Blank 2: the statistic scipy should recompute on every resample. Pass the function
# itself, as Step 10 passes np.mean: no brackets after the name.
# The two groups have different sizes, so each is resampled on its own
diff_result = bootstrap(
    (men, women),
    ____,
    n_resamples=B,
    batch=100,
    method="BCa",
    random_state=np.random.default_rng(RANDOM_STATE),
)

lo, hi = diff_result.confidence_interval
print(f"\n95% BCa CI for the difference: [{lo:.2f}, {hi:.2f}] hours")
# lo <= 0 <= hi is True when 0 lies between lo and hi
print(f"Does the interval contain zero? {lo <= 0 <= hi}")

**Expected output:** 21,790 men with a mean of 42.43 hours, 10,771 women with a mean of 36.41
hours, and an observed difference of 6.02 hours. The 95% BCa interval is [5.73, 6.28] hours,
which does not contain zero. The cell takes a few seconds. *NameError: name '____' is not
defined* means a blank is still empty.

### Interpretation Questions

1. Your interval for the difference in hours almost certainly excludes zero. Does that establish that being female *causes* fewer reported hours? Which chapter's machinery would you need to make a causal claim?
2. This is 1994 CPS data. Name one reason the interval understates your true uncertainty about the gap in the workforce today — and note that no amount of resampling would fix it.
3. Chapter 7 lists five situations where the bootstrap fails. Which one would bite hardest if you ran this same comparison on 40 workers from a single firm?

*Your answers here:*

---

## Extension (Optional — 15 min): Does a 95% Interval Actually Cover 95%?

An interval's advertised confidence level is a claim that can be checked. Simulate many
experiments where the true value is known and count how often each method's interval
contains it.

Two results come out of this, and the second is the one worth remembering:

- For the **mean**, the bootstrap buys you almost nothing. The percentile bootstrap and the
  CLT interval are both *first-order accurate*, so on skewed data at n = 50 they under-cover
  by about the same amount. The bootstrap is not a cure for skewness when a formula exists.
- For the **median** there is no parametric formula, so analysts reach for the interval they
  do have — the one built around the mean. Watch what that substitution costs.

<!-- idiom-note -->
> ### New idiom — dictionaries
>
> A **dictionary** is a lookup table: **keys** on the left of each colon,
> **values** on the right, wrapped in curly braces. Where a list answers
> *"what is in position 2"*, a dictionary answers *"what is stored under this
> name"*. Step 12 keeps four counters in one, and each key is a pair of words:
>
> ```python
> cover = {("mean", "parametric"): 0, ("mean", "bootstrap"): 0}
> cover[("mean", "parametric")] += 1       # add 1 to the counter under that key
> cover[("mean", "parametric")]            # 1: look up by KEY, not by position
> for (target, method), hits in cover.items():   # walk each key with its value
> ```
>
> `.items()` hands back each key with its value. Here the key is itself a pair,
> so `(target, method)` unpacks it into its two words.

In [ ]:
# EXTENSION — run as-is (it takes about 20 seconds)
# Step 12: measure the coverage each method actually delivers
true_mean = np.exp(3.5 + 0.5**2 / 2)      # mean of lognormal(3.5, 0.5); ** is "to the power of"
true_median = np.exp(3.5)                 # median of the same distribution
n_experiments = 400
n_sample = 50

# One counter per target and method; the note above shows how the pairs work
cover = {("mean", "parametric"): 0, ("mean", "bootstrap"): 0,
         ("median", "parametric"): 0, ("median", "bootstrap"): 0}
sim_rng = np.random.default_rng(RANDOM_STATE)

for i in range(n_experiments):
    s = sim_rng.lognormal(3.5, 0.5, n_sample)     # one experiment: 50 draws with a known truth

    # The one interval an analyst has a closed-form formula for
    se_i = s.std(ddof=1) / np.sqrt(n_sample)
    lo_p, hi_p = s.mean() - 1.96 * se_i, s.mean() + 1.96 * se_i

    # A different seed for each bootstrap, so the run is still reproducible
    r_mean = bootstrap((s,), np.mean, n_resamples=499, method="percentile",
                       random_state=np.random.default_rng(1_000 + i))
    r_med = bootstrap((s,), np.median, n_resamples=499, method="percentile",
                      random_state=np.random.default_rng(5_000 + i))

    # a <= x <= b is True when x lies between a and b, and += counts True as 1
    cover[("mean", "parametric")] += lo_p <= true_mean <= hi_p
    cover[("mean", "bootstrap")] += (r_mean.confidence_interval.low <= true_mean
                                   <= r_mean.confidence_interval.high)
    # The SAME parametric interval, now aimed at the median
    cover[("median", "parametric")] += lo_p <= true_median <= hi_p
    cover[("median", "bootstrap")] += (r_med.confidence_interval.low <= true_median
                                     <= r_med.confidence_interval.high)


def mc_se(p):
    """Monte Carlo SE of a coverage share: how much of a gap is simulation noise."""
    return (p * (1 - p) / n_experiments) ** 0.5      # ** 0.5 is the square root


print(f"True mean {true_mean:.2f}, true median {true_median:.2f}. Nominal level 95%.\n")
print(f"{'target':>8} {'method':>12} {'coverage':>10} {'MC SE':>8}")
print("-" * 42)
for (target, method), hits in cover.items():
    p = hits / n_experiments
    print(f"{target:>8} {method:>12} {p:>9.1%} {mc_se(p):>8.1%}")

**Expected output:** a true mean of 37.52 and a true median of 33.12. Compare every gap with the
MC SE column before calling it real.

- **The mean.** The parametric interval covers 92.5% and the percentile bootstrap 91.5%. The
  1-point gap is inside the Monte Carlo SE of about 1.3 points, so the two methods cannot be
  told apart here. Both fall slightly short of 95%: both are first-order accurate, and at
  n = 50 the CLT has not fully arrived.
- **The median.** The parametric interval covers only 66.8%. It is centred on the sample mean,
  which sits well above the median on right-skewed data, so it misses far more often than its
  label admits. The bootstrap covers 93.2%, because it is built around the statistic you
  actually asked about.

---

## Challenge (Take-Home): An Estimate With No Formula

The bootstrap's real value is statistics that have no closed-form standard error at all.
The **90/10 ratio** — the 90th percentile of income divided by the 10th — is a standard
inequality measure, and there is no `s / sqrt(n)` for it. Here you bootstrap it for the
positive capital gains in `cap_gain`.

**One wrinkle, and it is the only hard part.** `vectorized=True` lets scipy send your function
a 2-D batch of resamples, one resample per row, as well as the plain 1-D sample. `axis=-1`
means "the last axis": the only axis of a 1-D array, and the within-resample axis of a 2-D
batch, so one argument covers both cases. Leave `axis=axis` out of a percentile call and no
error warns you: that call takes one percentile of the whole batch instead of one per resample,
and the interval comes out wrong. `batch=100` is there for the memory reason Step 10 gave.

In [ ]:
# YOUR TASK — fill in the two blanks (____)
# Challenge: bootstrap an inequality measure
def ratio_90_10(x, axis=-1):
    """Return the 90th percentile divided by the 10th, computed along `axis`."""
    p90 = np.percentile(x, 90, axis=axis)
    p10 = ____          # the 10th percentile, same array, same axis
    return ____         # the ratio


# A 95% BCa interval for the ratio
res_ratio = bootstrap(
    (cap_gain,), ratio_90_10,
    n_resamples=B, batch=100, vectorized=True, axis=-1,
    confidence_level=0.95, method="BCa",
    random_state=np.random.default_rng(RANDOM_STATE),
)

lo_r, hi_r = res_ratio.confidence_interval
print(f"90/10 ratio of positive capital gains: {ratio_90_10(cap_gain):.2f}")
print(f"95% BCa interval: [{lo_r:.2f}, {hi_r:.2f}]   width = {hi_r - lo_r:.2f}")

**Expected output:** a 90/10 ratio of 6.45, and a 95% BCa interval of [6.24, 8.37], width 2.13.
*NameError: name '____' is not defined* means a blank is still empty.

**Reflect.** The interval is wide, and it is not symmetric around the point estimate. Chapter 7
Section 7.6 lists a failure mode that applies directly to extreme quantiles. Name it, say which
end of this interval you trust less, and say whether you would hand this number to a
policymaker without a caveat.

*Your answer here:*

---

## AI-Assisted Expansion: a bootstrap function written by AI, checked by you

In Step 8 you ran a bootstrap by hand: a loop that resamples `cap_gain` 2,000 times, takes the
mean of each resample, and reads off the 2.5th and 97.5th percentiles. In Step 10 scipy did the
same job. Now ask an AI to write that bootstrap as a reusable function, and check it against
your own two intervals. This is a fair test of AI: it writes the function in seconds, and you
can tell whether the function is right because you already know the answer.

**What to do**
1. Copy **prompt version 1** below into Claude ([claude.northeastern.edu](https://claude.northeastern.edu)) or ChatGPT.
2. Read what comes back before you run anything. The prompt leaves at least one
   important thing unsaid, so the AI had to guess. Find one guess that is wrong,
   or one that you would rather decide yourself. Pasting the version 1 function into
   the empty code cell and running the check is one way to find it.
3. Write **prompt version 2** in the cell below, fixing that one thing, and ask
   again. Paste the function from version 2 into the empty code cell and run the
   check.
4. Answer the questions at the end. They are what is graded, not the code.

**Reading the check.** It calls the AI's function twice with the same inputs and no seed, and
shows both intervals. Then, if the function has an argument called `seed`, it makes one more
call with `seed=42` (the lab's `RANDOM_STATE`) and shows it beside Step 10. The check passes
the seed as `seed=42`, so name the argument `seed`.

Steps 8 and 10 are both correct intervals for the same mean, yet their lower ends differ by
\$59 and their upper ends by \$16, because they drew different random resamples. A gap of
that size is what "right method, different random numbers" looks like. A function that draws
its resamples exactly as Step 10 does matches Step 10 to the dollar.

In [ ]:
# PROMPT, version 1 — copy it exactly as it is
#
# I have a NumPy array called cap_gain of positive capital gains, in dollars.
# Write a Python function bootstrap_ci(data, statistic, n_boot) that resamples
# data with replacement n_boot times, applies statistic to each resample, and
# returns the 95% percentile bootstrap interval as a tuple (lower, upper).
# Use NumPy only, and comment each line.

# PROMPT, version 2 — your revision. Keep everything from version 1 that was
# fine, and add or change the part the AI had to guess.
#
#

In [ ]:
# Paste the bootstrap_ci function from your version 2 here, then run the check below.

In [ ]:
# CHECK — compares the AI's function with your own intervals from Steps 8 and 10
# globals() holds every name defined so far in this notebook
if "bootstrap_ci" not in globals():
    print("Paste the AI's bootstrap_ci function in the cell above first.")
else:
    step8_lo, step8_hi = np.percentile(boot_means, 2.5), np.percentile(boot_means, 97.5)
    # The same call as Step 10 with method="percentile", so the same interval
    step10_lo, step10_hi = bootstrap((cap_gain,), np.mean, n_resamples=B, batch=100,
                                     method="percentile",
                                     random_state=np.random.default_rng(RANDOM_STATE)).confidence_interval
    print("                          lower      upper")
    print(f"Step 8, your loop      ${step8_lo:>9,.0f}  ${step8_hi:>9,.0f}")
    print(f"Step 10, scipy         ${step10_lo:>9,.0f}  ${step10_hi:>9,.0f}")

    # try runs the lines under it; if they raise a TypeError (a call with arguments the
    # function does not accept), the lines under except run instead of stopping the cell

    # First, two calls with the same inputs and no seed
    try:
        lo_1, hi_1 = bootstrap_ci(cap_gain, np.mean, B)
        lo_2, hi_2 = bootstrap_ci(cap_gain, np.mean, B)
        print(f"AI, no seed, call 1    ${lo_1:>9,.0f}  ${hi_1:>9,.0f}")
        print(f"AI, no seed, call 2    ${lo_2:>9,.0f}  ${hi_2:>9,.0f}")
    except TypeError:
        print("AI, no seed:           your function cannot be called without a seed.")

    # Then one call with the lab's seed, if the function has an argument called seed
    try:
        lo_s, hi_s = bootstrap_ci(cap_gain, np.mean, B, seed=RANDOM_STATE)
        print(f"AI, seed={RANDOM_STATE}            ${lo_s:>9,.0f}  ${hi_s:>9,.0f}")
    except TypeError:
        print("\nYour function has no argument called seed, so the check could not "
              "give it the lab's seed.")

    print("\nIf a pair differs, find out why before you trust the function.")

### Document the interaction

This is what is graded for the expansion.

1. **Your revision.** What did you change between version 1 and version 2, and
   what did the AI get wrong or have to guess that made you change it?
2. **The check.** Do the AI's two calls give the same interval, and does it match
   Step 10's? If a pair did not match at first, what was the cause? How do you know
   the method was not wrong? (Step 8 against Step 10 is a clue.)
3. **In your own words.** Pick the two or three lines of the AI's function that draw
   the resamples and read off the interval. Explain in 2–3 sentences what they do,
   and point to the part of Step 8 that does the same job.
4. **Would you use AI for this again?** One sentence: when is it worth it, and
   when is it faster to write the function yourself?

*Your answers here:*

1.
2.
3.
4.

---
## Digital Portfolio: Your README

Copy the prompt below into Claude or ChatGPT. Fill in each [YOUR VALUE] from your own output
first, and do not quote a number this notebook did not print. **Do NOT ask the AI to write
Python code — only documentation.**

```
I need help writing a project README for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Loaded the UCI Adult Census Income data (32,561 respondents)
* Separated the standard deviation of the data from the standard
  error of an estimate, and checked the s/sqrt(n) relationship
  across four sample sizes
* Built parametric confidence intervals from the CLT and showed
  the CLT converges quickly for age and slowly for capital gains
* Ran the bootstrap by hand, then with scipy.stats.bootstrap
* Found a degenerate case: the bootstrap SE of the median of 32,561
  whole-number ages is exactly 0.0000. Every resample returns the same
  median, because with that many ties the median is pinned. The
  bootstrap is reporting a real property of the data, not failing
* Compared percentile and BCa intervals on a heavily skewed variable
* Bootstrapped a two-sample difference in mean weekly hours
* Asked an AI to write a bootstrap-interval function, revised my prompt
  once, and checked the function against my own intervals
* Ran a coverage simulation on lognormal data at n=50. For the MEAN,
  the nominal 95% parametric interval covered [YOUR VALUE]% and the
  percentile bootstrap [YOUR VALUE]%, a gap inside the Monte Carlo
  standard error of about 1.3 points, so the two methods are
  indistinguishable here and both under-cover slightly. For the
  MEDIAN, the mean-centred parametric interval covered only about
  [YOUR VALUE]% while the bootstrap held near its nominal level

**Please write a README.md entry including:**
1. Project Title: Estimation & Bootstrapping — Quantifying Uncertainty Without a Formula
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'robust', 'comprehensive'
or 'leverage'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ3916-lab07-bootstrapping`

**First time?** Read the **GitHub Setup** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ3916-lab07-bootstrapping`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ3916-lab07-bootstrapping`, branch `main`,
   commit message `Lab 07 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.